In [1]:
from openai import OpenAI
openai_client = OpenAI()

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [3]:
import os
import sys
from pathlib import Path
from dotenv import find_dotenv

key = os.getenv("OPENAI_API_KEY")

print("Python:", sys.executable)
print("Working directory:", os.getcwd())
print("Dotenv file:", find_dotenv(usecwd=True))
print("Key exists:", bool(key))
print("Key length:", len(key) if key else None)
print("Key ending:", key[-4:] if key else None)

Python: /Users/patrickchong/Desktop/llm-zoomcamp-intro-to-RAG/.venv/bin/python
Working directory: /Users/patrickchong/Desktop/llm-zoomcamp-intro-to-RAG
Dotenv file: /Users/patrickchong/Desktop/llm-zoomcamp-intro-to-RAG/.env
Key exists: True
Key length: 164
Key ending: S4cA


In [13]:
question = "What is the exact percentage score required on CS50 problem sets to get a certificate?"
answer = llm(question)
print(answer)

There isn’t a required **percentage score on CS50 problem sets** to get the certificate.

For **CS50x**, the certificate is typically earned by completing the course requirements, which include:

- completing all **problem sets**,
- finishing the **final project**,
- and meeting the course’s overall submission/grade requirements.

CS50 generally does **not** award the certificate based on a single percentage threshold on problem sets alone. If you mean a specific CS50 variant (like **CS50x, CS50P, CS50AI**, etc.), I can tell you the exact certificate rules for that course.


In [18]:
# TODO: fill in the rest of detials here so I can run the below to show in HOOK.

from minsearch import Index
import json

# 1. Load the local JSON file directly
with open("cs50_faq.json", "r", encoding="utf-8") as f:
    documents = json.load(f)

# 2. Check how many documents were loaded
len(documents)

index = Index(
    text_fields=["question", "answer"],
    keyword_fields=["category"]
)

index.fit(documents)

index.search(question)  
index.search(question, filter_dict={'category':'Certificates'})  # returns all the questions only from the Certificates section


# The instructions tell the LLM its role and how to answer:

INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""



def search(question, category='Certificates'):
    boost_dict={'question':2.0}
    filter_dict={'category':category}
    
    return index.search(question,
                        boost_dict=boost_dict,
                        filter_dict=filter_dict,
                        num_results=5)

search_results = search(question)  # note that this search() function is the minsearch's Index() function and not
                                   # the search() function we defined just above.

def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["category"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text


prompt = build_prompt(question, search_results)
question = 'What is the exact percentage score required on CS50 problem sets to get a certificate?' 


In [15]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [16]:
# TO SHOW at the start - FULL RAG
def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer


In [20]:
answer = rag("What is the exact percentage score required on the CS50 problem sets to get a certificate?")
answer

'You need at least **70%** on each CS50 problem set to be eligible for a certificate.'